# Weibull Accelerated Failure Time (AFT) Model
## 1. Environment Setup

Import the required Python libraries and define file paths for the historical National Bridge Inventory (NBI) datasets.

In [1]:
# Import libraries for file handling and data analysis.
from pathlib import Path
from zipfile import ZipFile
import pandas as pd
import numpy as np

# Locate the ORIGINAL bridge-deterioration project directory.
PROJECT = Path.cwd().resolve()

if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent

if PROJECT.name == "bridge_time_primary_notebooks":
    PROJECT = PROJECT.parent

# Define paths for raw, processed and final datasets.
RAW = PROJECT / "data" / "raw"
PROCESSED = PROJECT / "data" / "processed"
FINAL = PROJECT / "data" / "final"

# Create output directories if they do not already exist.
PROCESSED.mkdir(parents=True, exist_ok=True)
FINAL.mkdir(parents=True, exist_ok=True)

# Identify all downloaded NBI ZIP archives.
zip_files = sorted(RAW.glob("*.zip"))

# Verify that the project directory and datasets are accessible.
print("Project directory:", PROJECT)
print("Raw data exists:", RAW.exists())
print("Final data exists:", FINAL.exists())
print("Number of NBI archives:", len(zip_files))

Project directory: /Users/annie/Downloads/bridge-deterioration
Raw data exists: True
Final data exists: True
Number of NBI archives: 34


In [2]:
from pathlib import Path

print("Current working directory:", Path.cwd())
print("RAW directory:", RAW)
print("RAW exists:", RAW.exists())

if RAW.exists():
    print("Files in RAW:", list(RAW.iterdir())[:10])

Current working directory: /Users/annie/Downloads/bridge-deterioration/bridge_time_primary_notebooks/notebooks
RAW directory: /Users/annie/Downloads/bridge-deterioration/data/raw
RAW exists: True
Files in RAW: [PosixPath('/Users/annie/Downloads/bridge-deterioration/data/raw/nbi_2024.zip'), PosixPath('/Users/annie/Downloads/bridge-deterioration/data/raw/nbi_2018.zip'), PosixPath('/Users/annie/Downloads/bridge-deterioration/data/raw/nbi_2019.zip'), PosixPath('/Users/annie/Downloads/bridge-deterioration/data/raw/nbi_2025.zip'), PosixPath('/Users/annie/Downloads/bridge-deterioration/data/raw/nbi_2009.zip'), PosixPath('/Users/annie/Downloads/bridge-deterioration/data/raw/nbi_2021.zip'), PosixPath('/Users/annie/Downloads/bridge-deterioration/data/raw/nbi_2020.zip'), PosixPath('/Users/annie/Downloads/bridge-deterioration/data/raw/nbi_2008.zip'), PosixPath('/Users/annie/Downloads/bridge-deterioration/data/raw/nbi_2022.zip'), PosixPath('/Users/annie/Downloads/bridge-deterioration/data/raw/nbi_2

## 2. Data Inspection

Examine the structure and column names of historical NBI datasets to identify inconsistencies before preprocessing.

In [3]:

# Select representative years across the historical dataset.
sample_years = [1992, 2000, 2010, 2020, 2025]

for year in sample_years:

    # Locate the ZIP archive for the selected year.
    zip_path = RAW / f"nbi_{year}.zip"

    # Open the archive without extracting the files.
    with ZipFile(zip_path) as archive:

        # Identify the TXT data file within the archive.
        txt_files = [
            name for name in archive.namelist()
            if name.lower().endswith(".txt")
        ]

        # Read the first five records, preserving values as strings.
        with archive.open(txt_files[0]) as file:
            df_sample = pd.read_csv(
                file,
                nrows=5,
                dtype=str,
                quotechar="'",
                encoding="latin-1"
            )

    # Standardise column names by removing surrounding whitespace.
    df_sample.columns = df_sample.columns.str.strip()

    # Display the year and available columns.
    print(f"\nYEAR: {year}")
    print(f"Number of columns: {len(df_sample.columns)}")
    print("Column names:")
    print(df_sample.columns.tolist())
    print("-" * 60)



YEAR: 1992
Number of columns: 134
Column names:
['STATE_CODE_001', 'STRUCTURE_NUMBER_008', 'RECORD_TYPE_005A', 'ROUTE_PREFIX_005B', 'SERVICE_LEVEL_005C', 'ROUTE_NUMBER_005D', 'DIRECTION_005E', 'HIGHWAY_DISTRICT_002', 'COUNTY_CODE_003', 'PLACE_CODE_004', 'FEATURES_DESC_006A', 'CRITICAL_FACILITY_006B', 'FACILITY_CARRIED_007', 'LOCATION_009', 'MIN_VERT_CLR_010', 'KILOPOINT_011', 'BASE_HWY_NETWORK_012', 'LRS_INV_ROUTE_013A', 'SUBROUTE_NO_013B', 'LAT_016', 'LONG_017', 'DETOUR_KILOS_019', 'TOLL_020', 'MAINTENANCE_021', 'OWNER_022', 'FUNCTIONAL_CLASS_026', 'YEAR_BUILT_027', 'TRAFFIC_LANES_ON_028A', 'TRAFFIC_LANES_UND_028B', 'ADT_029', 'YEAR_ADT_030', 'DESIGN_LOAD_031', 'APPR_WIDTH_MT_032', 'MEDIAN_CODE_033', 'DEGREES_SKEW_034', 'STRUCTURE_FLARED_035', 'RAILINGS_036A', 'TRANSITIONS_036B', 'APPR_RAIL_036C', 'APPR_RAIL_END_036D', 'HISTORY_037', 'NAVIGATION_038', 'NAV_VERT_CLR_MT_039', 'NAV_HORR_CLR_MT_040', 'OPEN_CLOSED_POSTED_041', 'SERVICE_ON_042A', 'SERVICE_UND_042B', 'STRUCTURE_KIND_043A', 

## 3. Data Cleaning and Preprocessing

Select and standardise the relevant NBI variables across historical datasets. Validate column availability, bridge identifiers, condition ratings and numerical features before constructing the survival analysis dataset.

In [4]:

# Define the NBI variables required for bridge identification,
# deterioration classification and AFT model predictors.
required_columns = [
    "STATE_CODE_001",
    "STRUCTURE_NUMBER_008",
    "YEAR_BUILT_027",
    "YEAR_RECONSTRUCTED_106",
    "ADT_029",
    "STRUCTURE_KIND_043A",
    "STRUCTURE_TYPE_043B",
    "STRUCTURE_LEN_MT_049",
    "DECK_COND_058",
    "SUPERSTRUCTURE_COND_059",
    "SUBSTRUCTURE_COND_060",
    "CULVERT_COND_062",
    "DATE_OF_INSPECT_090"
]

column_results = []

# Check the column names in every annual archive.
for year in range(1992, 2026):
    zip_path = RAW / f"nbi_{year}.zip"

    with ZipFile(zip_path) as archive:
        # Locate the TXT data file inside the archive.
        txt_name = next(
            name for name in archive.namelist()
            if name.lower().endswith(".txt")
        )

        # Read only the header to avoid loading the full dataset.
        with archive.open(txt_name) as file:
            columns = pd.read_csv(
                file,
                nrows=0,
                quotechar="'",
                encoding="latin-1"
            ).columns.str.strip()

    # Identify required variables missing from this year.
    missing_columns = [
        col for col in required_columns
        if col not in columns
    ]

    column_results.append({
        "year": year,
        "missing_columns": missing_columns,
        "all_present": len(missing_columns) == 0
    })

# Display a summary of column availability.
column_check = pd.DataFrame(column_results)

print("Years checked:", len(column_check))
print("Years with all required columns:",
      column_check["all_present"].sum())

display(column_check[
    ~column_check["all_present"]
])


Years checked: 34
Years with all required columns: 34


,year,missing_columns,all_present


In [5]:

# Define variables to retain from the original NBI datasets.
columns_to_keep = required_columns

# Define the condition rating columns.
condition_columns = [
    "DECK_COND_058",
    "SUPERSTRUCTURE_COND_059",
    "SUBSTRUCTURE_COND_060",
    "CULVERT_COND_062"
]

# Define variables that require numerical conversion.
numeric_columns = [
    "YEAR_BUILT_027",
    "YEAR_RECONSTRUCTED_106",
    "ADT_029",
    "STRUCTURE_LEN_MT_049"
]

# Process each historical dataset separately to reduce memory usage.
for year in range(1992, 2026):

    zip_path = RAW / f"nbi_{year}.zip"

    with ZipFile(zip_path) as archive:
        # Locate the TXT data file within the archive.
        txt_name = next(
            name for name in archive.namelist()
            if name.lower().endswith(".txt")
        )

        # Load only the variables required for analysis.
        with archive.open(txt_name) as file:
            df = pd.read_csv(
                file,
                usecols=columns_to_keep,
                dtype=str,
                quotechar="'",
                encoding="latin-1",
                low_memory=False
            )

    # Standardise column names and remove surrounding whitespace.
    df.columns = df.columns.str.strip()
    df = df.apply(lambda col: col.str.strip())

    # Convert blank strings into missing values.
    df = df.replace("", np.nan)

    # Preserve bridge identifiers as text, including leading zeros.
    df["bridge_id"] = (
        df["STATE_CODE_001"].fillna("") + "_" +
        df["STRUCTURE_NUMBER_008"].fillna("")
    )

    # Convert numerical predictors to numeric data types.
    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Convert condition ratings to numbers.
    # Non-applicable ratings such as 'N' become missing values.
    for col in condition_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

        # Only ratings from 0 to 9 are valid.
        df.loc[~df[col].between(0, 9), col] = np.nan

    # Determine condition using the lowest applicable rating.
    # Culvert ratings are used when the three bridge components are absent.
    bridge_rating = df[
        condition_columns[:3]
    ].min(axis=1)

    df["condition_rating"] = bridge_rating.fillna(
        df["CULVERT_COND_062"]
    )

    # Classify each structure as Good, Fair, Poor or Unknown.
    df["condition"] = np.select(
        [
            df["condition_rating"].between(7, 9),
            df["condition_rating"].between(5, 6),
            df["condition_rating"].between(0, 4)
        ],
        ["Good", "Fair", "Poor"],
        default="Unknown"
    )

    # Calculate bridge age at the observation year.
    df["bridge_age"] = year - df["YEAR_BUILT_027"]

    # Exclude implausible ages from the age predictor.
    df.loc[
        ~df["bridge_age"].between(0, 200),
        "bridge_age"
    ] = np.nan

    # Record the year of the annual dataset.
    df["observation_year"] = year

    # Save the cleaned annual dataset in Parquet format.
    output_path = PROCESSED / f"nbi_{year}_clean.parquet"
    df.to_parquet(output_path, index=False)

    # Display a processing summary.
    print(
        f"{year}: {len(df):,} records | "
        f"Good: {(df['condition'] == 'Good').sum():,} | "
        f"Fair: {(df['condition'] == 'Fair').sum():,} | "
        f"Poor: {(df['condition'] == 'Poor').sum():,} | "
        f"Unknown: {(df['condition'] == 'Unknown').sum():,}"
    )

print("\nAll annual datasets processed.")


1992: 666,201 records | Good: 278,435 | Fair: 215,357 | Poor: 90,114 | Unknown: 82,295


KeyboardInterrupt: 

In [ ]:

# Select representative years for data quality checks.
validation_years = [1992, 2009, 2010, 2025]

validation_results = []

for year in validation_years:

    # Load the cleaned annual dataset.
    file_path = PROCESSED / f"nbi_{year}_clean.parquet"
    df = pd.read_parquet(file_path)

    # Identify missing or incomplete bridge identifiers.
    missing_ids = (
        df["STATE_CODE_001"].isna() |
        df["STRUCTURE_NUMBER_008"].isna()
    ).sum()

    # Count duplicate identifiers within the same year.
    duplicate_ids = df.duplicated(
        subset=["STATE_CODE_001", "STRUCTURE_NUMBER_008"]
    ).sum()

    # Calculate missing values for important AFT predictors.
    missing_age = df["bridge_age"].isna().sum()
    missing_traffic = df["ADT_029"].isna().sum()
    missing_length = df["STRUCTURE_LEN_MT_049"].isna().sum()

    # Summarise the results for each year.
    validation_results.append({
        "Year": year,
        "Total Records": len(df),
        "Missing IDs": missing_ids,
        "Duplicate IDs": duplicate_ids,
        "Missing Age": missing_age,
        "Missing Traffic": missing_traffic,
        "Missing Length": missing_length,
        "Unknown Condition": (df["condition"] == "Unknown").sum()
    })

# Display the data quality summary.
validation_df = pd.DataFrame(validation_results)
display(validation_df)


,Year,Total Records,Missing IDs,Duplicate IDs,Missing Age,Missing Traffic,Missing Length,Unknown Condition
0,1992,666201,0,63691,2701,4414,1292,82295
1,2009,713113,0,90247,596,972,27,105286
2,2010,604492,0,37,322,65,0,377
3,2025,624193,0,2,49,0,0,0


In [ ]:

# Compare duplicate bridge identifiers before and after 2010.
for year in [2009, 2010]:

    # Load the cleaned dataset for the selected year.
    df = pd.read_parquet(
        PROCESSED / f"nbi_{year}_clean.parquet"
    )

    # Identify all records sharing a state and structure number.
    duplicate_mask = df.duplicated(
        subset=["STATE_CODE_001", "STRUCTURE_NUMBER_008"],
        keep=False
    )

    duplicate_records = df.loc[duplicate_mask].copy()

    # Count the number of distinct affected bridge identifiers.
    duplicate_groups = duplicate_records.groupby(
        ["STATE_CODE_001", "STRUCTURE_NUMBER_008"]
    ).ngroups

    print(f"\nYEAR: {year}")
    print("Records with duplicate IDs:", len(duplicate_records))
    print("Distinct duplicate IDs:", duplicate_groups)

    # Display sample duplicate records for comparison.
    display(
        duplicate_records[
            [
                "STATE_CODE_001",
                "STRUCTURE_NUMBER_008",
                "condition",
                "YEAR_BUILT_027",
                "STRUCTURE_LEN_MT_049",
                "ADT_029"
            ]
        ].head(10)
    )



YEAR: 2009
Records with duplicate IDs: 168228
Distinct duplicate IDs: 77981


,STATE_CODE_001,STRUCTURE_NUMBER_008,condition,YEAR_BUILT_027,STRUCTURE_LEN_MT_049,ADT_029
614,01,001008,Fair,1932.0,188.9,6490.0
823,01,001390,Fair,1936.0,217.3,4860.0
825,01,001392,Fair,1936.0,53.3,44710.0
831,01,001399,Fair,1936.0,257.9,33120.0
927,01,001553,Fair,1937.0,8.2,9100.0
1031,01,001709,Fair,1938.0,36.6,41790.0
1398,01,002243,Fair,1940.0,242.3,5895.0
1701,01,002648,Fair,1942.0,42.4,34930.0
1806,01,002800,Fair,1927.0,104.5,20250.0
2090,01,003202,Poor,1948.0,76.8,6100.0



YEAR: 2010
Records with duplicate IDs: 74
Distinct duplicate IDs: 37


,STATE_CODE_001,STRUCTURE_NUMBER_008,condition,YEAR_BUILT_027,STRUCTURE_LEN_MT_049,ADT_029
15953,01,CEPSAMAL0000002,Good,1978.0,39.6,220.0
15954,01,CEPSAMAL0000002,Good,1978.0,39.6,220.0
16980,02,00000000000E101,Poor,1959.0,39.5,50.0
16981,02,00000000000E101,Good,1985.0,15.3,200.0
37284,05,AHTD00000017769,Fair,1967.0,168.6,250.0
37285,05,AHTD00000017769,Good,1967.0,168.6,200.0
37286,05,AHTD000000M3472,Good,1965.0,312.1,1300.0
37287,05,AHTD000000M3472,Good,1965.0,95.1,1500.0
37288,05,AHTD000000M3512,Good,1964.0,87.8,5700.0
37289,05,AHTD000000M3512,Good,1964.0,87.8,5100.0


In [ ]:

# Inspect record types and duplicate identifiers in selected years.
for year in [2009, 2010]:

    zip_path = RAW / f"nbi_{year}.zip"

    with ZipFile(zip_path) as archive:
        txt_name = next(
            name for name in archive.namelist()
            if name.lower().endswith(".txt")
        )

        # Load identifiers and record classification fields.
        with archive.open(txt_name) as file:
            df = pd.read_csv(
                file,
                usecols=[
                    "STATE_CODE_001",
                    "STRUCTURE_NUMBER_008",
                    "RECORD_TYPE_005A",
                    "ROUTE_PREFIX_005B",
                    "ROUTE_NUMBER_005D"
                ],
                dtype=str,
                quotechar="'",
                encoding="latin-1",
                low_memory=False
            )

    # Remove surrounding whitespace from text values.
    df = df.apply(lambda col: col.str.strip())

    # Identify records with repeated bridge identifiers.
    duplicate_mask = df.duplicated(
        subset=["STATE_CODE_001", "STRUCTURE_NUMBER_008"],
        keep=False
    )

    print(f"\nYEAR: {year}")

    print("\nRecord type distribution:")
    print(df["RECORD_TYPE_005A"].value_counts(dropna=False))

    print("\nRecord types among duplicated identifiers:")
    print(
        df.loc[duplicate_mask, "RECORD_TYPE_005A"]
        .value_counts(dropna=False)
    )

    print("\nExample duplicated records:")
    display(df.loc[duplicate_mask].head(12))



YEAR: 2009

Record type distribution:
RECORD_TYPE_005A
1    608417
2     83407
A      8753
B      8318
C      2112
D      1002
E       441
F       227
G       133
H        82
I        50
J        35
K        25
L        22
M        16
N        14
O        10
Q         9
P         9
R         7
S         5
Z         5
Y         3
T         3
V         3
U         2
X         2
W         1
Name: count, dtype: int64

Record types among duplicated identifiers:
RECORD_TYPE_005A
1    77756
2    69249
A     8689
B     8316
C     2112
D     1002
E      441
F      227
G      133
H       82
I       50
J       35
K       25
L       22
M       16
N       14
O       10
Q        9
P        9
R        7
S        5
Z        5
Y        3
T        3
V        3
U        2
X        2
W        1
Name: count, dtype: int64

Example duplicated records:


,STATE_CODE_001,STRUCTURE_NUMBER_008,RECORD_TYPE_005A,ROUTE_PREFIX_005B,ROUTE_NUMBER_005D
614,01,001008,1,3,00155
823,01,001390,1,2,00031
825,01,001392,1,3,00005
831,01,001399,1,2,00011
927,01,001553,1,5,00000
1031,01,001709,1,2,00072
1398,01,002243,1,2,00031
1701,01,002648,1,2,00031
1806,01,002800,1,3,00215
2090,01,003202,1,3,00012



YEAR: 2010

Record type distribution:
RECORD_TYPE_005A
1    604492
Name: count, dtype: int64

Record types among duplicated identifiers:
RECORD_TYPE_005A
1    74
Name: count, dtype: int64

Example duplicated records:


,STATE_CODE_001,STRUCTURE_NUMBER_008,RECORD_TYPE_005A,ROUTE_PREFIX_005B,ROUTE_NUMBER_005D
15953,01,CEPSAMAL0000002,1,7,00000
15954,01,CEPSAMAL0000002,1,7,00000
16980,02,00000000000E101,1,6,01002
16981,02,00000000000E101,1,6,01004
37284,05,AHTD00000017769,1,4,00096
37285,05,AHTD00000017769,1,4,00096
37286,05,AHTD000000M3472,1,3,00187
37287,05,AHTD000000M3472,1,3,00187
37288,05,AHTD000000M3512,1,3,00025
37289,05,AHTD000000M3512,1,3,00025


In [ ]:

# Compare record counts and duplicate identifiers after selecting type 1.
record_type_results = []

for year in range(1992, 2026):

    zip_path = RAW / f"nbi_{year}.zip"

    with ZipFile(zip_path) as archive:
        txt_name = next(
            name for name in archive.namelist()
            if name.lower().endswith(".txt")
        )

        # Load only the fields required for record type validation.
        with archive.open(txt_name) as file:
            df = pd.read_csv(
                file,
                usecols=[
                    "STATE_CODE_001",
                    "STRUCTURE_NUMBER_008",
                    "RECORD_TYPE_005A"
                ],
                dtype=str,
                quotechar="'",
                encoding="latin-1"
            )

    # Standardise identifier and record type values.
    df = df.apply(lambda col: col.str.strip())

    # Retain primary record type for diagnostic comparison.
    primary = df[df["RECORD_TYPE_005A"] == "1"]

    # Count repeated identifiers among primary records.
    duplicate_count = primary.duplicated(
        subset=["STATE_CODE_001", "STRUCTURE_NUMBER_008"]
    ).sum()

    record_type_results.append({
        "Year": year,
        "All Records": len(df),
        "Type 1 Records": len(primary),
        "Other Records": len(df) - len(primary),
        "Type 1 Duplicates": duplicate_count
    })

record_type_check = pd.DataFrame(record_type_results)

display(record_type_check)


,Year,All Records,Type 1 Records,Other Records,Type 1 Duplicates
0,1992,666201,584826,81375,0
1,1993,668429,586141,82288,0
2,1994,670872,588545,82327,0
3,1995,680657,596438,84219,954
4,1996,679056,594155,84901,9
5,1997,683040,596629,86411,101
6,1998,683855,596137,87718,98
7,1999,688409,598741,89668,344
8,2000,691053,600295,90758,352
9,2001,694936,602948,91988,93


In [ ]:

# Examine remaining duplicate identifiers among type 1 records.
for year in [1995, 2000, 2009, 2010, 2025]:

    zip_path = RAW / f"nbi_{year}.zip"

    with ZipFile(zip_path) as archive:
        txt_name = next(
            name for name in archive.namelist()
            if name.lower().endswith(".txt")
        )

        # Load the original records and retain relevant variables.
        with archive.open(txt_name) as file:
            df = pd.read_csv(
                file,
                usecols=required_columns + ["RECORD_TYPE_005A"],
                dtype=str,
                quotechar="'",
                encoding="latin-1",
                low_memory=False
            )

    # Standardise text values and select type 1 records.
    df = df.apply(lambda col: col.str.strip())
    df = df[df["RECORD_TYPE_005A"] == "1"].copy()

    # Identify repeated bridge identifiers.
    duplicate_mask = df.duplicated(
        subset=["STATE_CODE_001", "STRUCTURE_NUMBER_008"],
        keep=False
    )

    duplicates = df.loc[duplicate_mask]

    # Identify records that are identical across all selected variables.
    exact_duplicates = duplicates.duplicated(
        subset=required_columns
    ).sum()

    print(f"\nYEAR: {year}")
    print("Records with repeated IDs:", len(duplicates))
    print("Exact duplicate records:", exact_duplicates)

    # Display a small sample of remaining duplicates.
    display(
        duplicates[
            [
                "STATE_CODE_001",
                "STRUCTURE_NUMBER_008",
                "YEAR_BUILT_027",
                "DECK_COND_058",
                "SUPERSTRUCTURE_COND_059",
                "SUBSTRUCTURE_COND_060",
                "CULVERT_COND_062"
            ]
        ].head(8)
    )



YEAR: 1995
Records with repeated IDs: 1908
Exact duplicate records: 392


,STATE_CODE_001,STRUCTURE_NUMBER_008,YEAR_BUILT_027,DECK_COND_058,SUPERSTRUCTURE_COND_059,SUBSTRUCTURE_COND_060,CULVERT_COND_062
16099,02,1390,1985,8,8,8,N
16100,02,1390,1985,8,8,8,N
18572,04,3006,1966,4,8,8,N
18573,04,3006,1966,5,8,8,N
18574,04,3007,1964,7,7,7,N
18575,04,3007,1964,7,8,8,N
18576,04,3008,1930,6,7,8,N
18577,04,3008,1930,6,7,8,N



YEAR: 2000
Records with repeated IDs: 703
Exact duplicate records: 23


,STATE_CODE_001,STRUCTURE_NUMBER_008,YEAR_BUILT_027,DECK_COND_058,SUPERSTRUCTURE_COND_059,SUBSTRUCTURE_COND_060,CULVERT_COND_062
23348,04,03199,1935,6,6,6,N
23349,04,03200,1934,7,7,7,N
23350,04,03201,1950,6,6,6,N
23351,04,03202,1900,7,7,7,N
23352,04,03203,1900,7,7,7,N
23353,04,03204,1931,5,4,4,N
23354,04,03206,1900,6,7,7,N
23355,04,03207,1900,6,6,7,N



YEAR: 2009
Records with repeated IDs: 190
Exact duplicate records: 2


,STATE_CODE_001,STRUCTURE_NUMBER_008,YEAR_BUILT_027,DECK_COND_058,SUPERSTRUCTURE_COND_059,SUBSTRUCTURE_COND_060,CULVERT_COND_062
15928,01,CEPSAMAL0000002,1978,8,8,8,N
15929,01,CEPSAMAL0000002,1978,8,8,8,N
24559,04,00000000000N628,1940,0,2,3,N
24560,04,00000000000N628,1940,0,2,3,N
37205,05,7330003P0000000,1955,6,7,7,N
37206,05,7330003P0000000,1955,7,8,7,N
37210,05,AHTD00000017769,1967,5,7,7,N
37211,05,AHTD00000017769,1967,7,8,8,N



YEAR: 2010
Records with repeated IDs: 74
Exact duplicate records: 0


,STATE_CODE_001,STRUCTURE_NUMBER_008,YEAR_BUILT_027,DECK_COND_058,SUPERSTRUCTURE_COND_059,SUBSTRUCTURE_COND_060,CULVERT_COND_062
15953,01,CEPSAMAL0000002,1978,8,8,8,N
15954,01,CEPSAMAL0000002,1978,8,8,8,N
16980,02,00000000000E101,1959,N,N,N,3
16981,02,00000000000E101,1985,7,7,7,N
37284,05,AHTD00000017769,1967,5,7,5,N
37285,05,AHTD00000017769,1967,7,8,8,N
37286,05,AHTD000000M3472,1965,7,8,8,N
37287,05,AHTD000000M3472,1965,7,8,8,N



YEAR: 2025
Records with repeated IDs: 4
Exact duplicate records: 0


,STATE_CODE_001,STRUCTURE_NUMBER_008,YEAR_BUILT_027,DECK_COND_058,SUPERSTRUCTURE_COND_059,SUBSTRUCTURE_COND_060,CULVERT_COND_062
247562,25,N100108PDDOTNBI,2009,7,6,6,N
247563,25,N100108PDDOTNBI,2007,7,8,8,N
574872,51,000000000001826,1951,6,4,5,N
574873,51,000000000001826,1968,N,N,N,6


In [ ]:

# Assess duplicate identifier conflicts across all annual datasets.
conflict_results = []

for year in range(1992, 2026):

    # Load the cleaned annual dataset.
    df = pd.read_parquet(
        PROCESSED / f"nbi_{year}_clean.parquet"
    )

    # Identify bridge identifiers with multiple records.
    id_columns = ["STATE_CODE_001", "STRUCTURE_NUMBER_008"]

    duplicate_records = df[
        df.duplicated(subset=id_columns, keep=False)
    ].copy()

    # Count identifiers associated with conflicting condition ratings.
    condition_counts = duplicate_records.groupby(
        id_columns
    )["condition_rating"].nunique(dropna=False)

    conflicting_ids = (condition_counts > 1).sum()

    # Count all identifiers containing multiple records.
    duplicate_ids = duplicate_records.groupby(
        id_columns
    ).ngroups

    conflict_results.append({
        "Year": year,
        "Duplicate IDs": duplicate_ids,
        "Conflicting Condition IDs": conflicting_ids
    })

# Display duplicate and condition conflict counts.
conflict_df = pd.DataFrame(conflict_results)

display(conflict_df)

print(
    "\nTotal conflicting bridge-year IDs:",
    conflict_df["Conflicting Condition IDs"].sum()
)


,Year,Duplicate IDs,Conflicting Condition IDs
0,1992,59196,58834
1,1993,59628,59269
2,1994,60388,60036
3,1995,62742,61534
4,1996,62104,61681
5,1997,63948,63414
6,1998,64716,64246
7,1999,66870,66001
8,2000,67478,66559
9,2001,68191,67516



Total conflicting bridge-year IDs: 1218889


In [ ]:

# Apply consistent record selection and deduplication across all years.
cleaning_results = []

for year in range(1992, 2026):

    zip_path = RAW / f"nbi_{year}.zip"

    with ZipFile(zip_path) as archive:
        txt_name = next(
            name for name in archive.namelist()
            if name.lower().endswith(".txt")
        )

        # Load the original data with the record type field.
        with archive.open(txt_name) as file:
            df = pd.read_csv(
                file,
                usecols=required_columns + ["RECORD_TYPE_005A"],
                dtype=str,
                quotechar="'",
                encoding="latin-1",
                low_memory=False
            )

    # Standardise text values and missing entries.
    df.columns = df.columns.str.strip()
    df = df.apply(lambda col: col.str.strip())
    df = df.replace("", np.nan)

    original_count = len(df)

    # Retain primary inventory records.
    df = df[df["RECORD_TYPE_005A"] == "1"].copy()
    primary_count = len(df)

    # Remove bridge-year identifiers with multiple type 1 records.
    id_columns = ["STATE_CODE_001", "STRUCTURE_NUMBER_008"]

    duplicate_mask = df.duplicated(
        subset=id_columns,
        keep=False
    )

    ambiguous_count = duplicate_mask.sum()
    df = df.loc[~duplicate_mask].copy()

    # Create a consistent bridge identifier.
    df["bridge_id"] = (
        df["STATE_CODE_001"] + "_" +
        df["STRUCTURE_NUMBER_008"]
    )

    # Convert numerical predictors.
    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Validate condition ratings.
    for col in condition_columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df.loc[~df[col].between(0, 9), col] = np.nan

    # Calculate the lowest applicable bridge component rating.
    bridge_rating = df[
        [
            "DECK_COND_058",
            "SUPERSTRUCTURE_COND_059",
            "SUBSTRUCTURE_COND_060"
        ]
    ].min(axis=1)

    df["condition_rating"] = bridge_rating.fillna(
        df["CULVERT_COND_062"]
    )

    # Classify bridge condition.
    df["condition"] = np.select(
        [
            df["condition_rating"].between(7, 9),
            df["condition_rating"].between(5, 6),
            df["condition_rating"].between(0, 4)
        ],
        ["Good", "Fair", "Poor"],
        default="Unknown"
    )

    # Calculate bridge age at the observation year.
    df["bridge_age"] = year - df["YEAR_BUILT_027"]
    df.loc[
        ~df["bridge_age"].between(0, 200),
        "bridge_age"
    ] = np.nan

    df["observation_year"] = year

    # Save the updated annual dataset.
    output_path = PROCESSED / f"nbi_{year}_clean.parquet"
    df.to_parquet(output_path, index=False)

    cleaning_results.append({
        "Year": year,
        "Original Records": original_count,
        "Type 1 Records": primary_count,
        "Ambiguous Records Removed": ambiguous_count,
        "Final Records": len(df)
    })

# Display the cleaning summary.
cleaning_summary = pd.DataFrame(cleaning_results)
display(cleaning_summary)

print(
    "\nTotal cleaned records:",
    f"{cleaning_summary['Final Records'].sum():,}"
)


,Year,Original Records,Type 1 Records,Ambiguous Records Removed,Final Records
0,1992,666201,584826,0,584826
1,1993,668429,586141,0,586141
2,1994,670872,588545,0,588545
3,1995,680657,596438,1908,594530
4,1996,679056,594155,18,594137
5,1997,683040,596629,202,596427
6,1998,683855,596137,196,595941
7,1999,688409,598741,688,598053
8,2000,691053,600295,703,599592
9,2001,694936,602948,186,602762



Total cleaned records: 20,646,344


## 4. Survival Dataset Construction

Link historical observations using unique bridge identifiers to construct longitudinal records. Identify bridges initially classified as Good, their subsequent deterioration events, and right-censored observations.

In [ ]:

# Select the variables required to examine historical bridge observations.
history_columns = [
    "bridge_id",
    "observation_year",
    "condition",
    "condition_rating",
    "bridge_age",
    "YEAR_BUILT_027",
    "YEAR_RECONSTRUCTED_106",
    "ADT_029",
    "STRUCTURE_KIND_043A",
    "STRUCTURE_TYPE_043B",
    "STRUCTURE_LEN_MT_049"
]

# Load the relevant variables from all cleaned annual datasets.
annual_frames = []

for year in range(1992, 2026):
    file_path = PROCESSED / f"nbi_{year}_clean.parquet"

    df_year = pd.read_parquet(
        file_path,
        columns=history_columns
    )

    annual_frames.append(df_year)

# Combine the annual records into a longitudinal dataset.
bridge_history = pd.concat(
    annual_frames,
    ignore_index=True
)

# Release the temporary annual DataFrames.
del annual_frames, df_year

# Summarise the combined historical data.
print("Total bridge-year observations:", f"{len(bridge_history):,}")
print("Unique bridge identifiers:", f"{bridge_history['bridge_id'].nunique():,}")

print("\nCondition distribution:")
print(bridge_history["condition"].value_counts())

print("\nObservation year range:")
print(
    bridge_history["observation_year"].min(),
    "to",
    bridge_history["observation_year"].max()
)


Total bridge-year observations: 20,646,344
Unique bridge identifiers: 1,039,107

Condition distribution:
condition
Good       9696487
Fair       8827290
Poor       2092295
Unknown      30272
Name: count, dtype: int64

Observation year range:
1992 to 2025


In [ ]:

# Summarise observation coverage for each bridge identifier.
bridge_coverage = bridge_history.groupby("bridge_id")[
    "observation_year"
].agg(
    first_year="min",
    last_year="max",
    years_observed="nunique"
)

# Calculate the observation window for each bridge.
bridge_coverage["observation_window"] = (
    bridge_coverage["last_year"] -
    bridge_coverage["first_year"] + 1
)

# Identify bridges with missing annual observations.
bridge_coverage["has_gaps"] = (
    bridge_coverage["years_observed"] <
    bridge_coverage["observation_window"]
)

# Summarise longitudinal data coverage.
print("Unique bridge identifiers:", f"{len(bridge_coverage):,}")

print(
    "Bridges observed in only one year:",
    f"{(bridge_coverage['years_observed'] == 1).sum():,}"
)

print(
    "Bridges observed in multiple years:",
    f"{(bridge_coverage['years_observed'] > 1).sum():,}"
)

print(
    "Bridges with gaps in annual records:",
    f"{bridge_coverage['has_gaps'].sum():,}"
)

print("\nYears observed per bridge:")
display(bridge_coverage["years_observed"].describe())


Unique bridge identifiers: 1,039,107
Bridges observed in only one year: 73,392
Bridges observed in multiple years: 965,715
Bridges with gaps in annual records: 17,650

Years observed per bridge:


count    1.039107e+06
mean     1.986931e+01
std      1.189897e+01
min      1.000000e+00
25%      9.000000e+00
50%      1.900000e+01
75%      3.400000e+01
max      3.400000e+01
Name: years_observed, dtype: float64

In [ ]:

# Arrange observations chronologically within each bridge identifier.
bridge_history = bridge_history.sort_values(
    ["bridge_id", "observation_year"]
).reset_index(drop=True)

# Identify the previous recorded condition and observation year.
bridge_history["previous_condition"] = (
    bridge_history.groupby("bridge_id")["condition"].shift(1)
)

bridge_history["previous_year"] = (
    bridge_history.groupby("bridge_id")["observation_year"].shift(1)
)

# Calculate the interval between consecutive recorded observations.
bridge_history["year_gap"] = (
    bridge_history["observation_year"] -
    bridge_history["previous_year"]
)

# Identify transitions from Good to Fair or Poor.
deterioration_mask = (
    (bridge_history["previous_condition"] == "Good") &
    (bridge_history["condition"].isin(["Fair", "Poor"]))
)

# Identify apparent improvements from Fair or Poor to Good.
improvement_mask = (
    bridge_history["previous_condition"].isin(["Fair", "Poor"]) &
    (bridge_history["condition"] == "Good")
)

# Count condition transitions across the historical records.
print(
    "Good to Fair/Poor transitions:",
    f"{deterioration_mask.sum():,}"
)

print(
    "Fair/Poor to Good transitions:",
    f"{improvement_mask.sum():,}"
)

print(
    "Good to Fair/Poor transitions with observation gaps:",
    f"{(deterioration_mask & (bridge_history['year_gap'] > 1)).sum():,}"
)

print(
    "Unique bridges with observed Good to Fair/Poor transitions:",
    f"{bridge_history.loc[deterioration_mask, 'bridge_id'].nunique():,}"
)

# Display example deterioration transitions.
display(
    bridge_history.loc[
        deterioration_mask,
        [
            "bridge_id",
            "previous_year",
            "observation_year",
            "year_gap",
            "previous_condition",
            "condition"
        ]
    ].head(10)
)


Good to Fair/Poor transitions: 421,496
Fair/Poor to Good transitions: 262,255
Good to Fair/Poor transitions with observation gaps: 1,320
Unique bridges with observed Good to Fair/Poor transitions: 354,698


,bridge_id,previous_year,observation_year,year_gap,previous_condition,condition
3,01_00000000000S700,1998.0,1999,1.0,Good,Poor
33,01_00000000000S702,2021.0,2022,1.0,Good,Fair
53,01_00000000000S703,2021.0,2022,1.0,Good,Fair
65,01_0000000000M0022,2005.0,2006,1.0,Good,Fair
11183,01_000001014002450,2003.0,2004,1.0,Good,Fair
13871,01_000004,2008.0,2009,1.0,Good,Poor
14355,01_000007,2006.0,2007,1.0,Good,Fair
14406,01_000008,2022.0,2023,1.0,Good,Fair
14466,01_00000B253040000,2008.0,2009,1.0,Good,Fair
14501,01_00000BLN0BR0010,2014.0,2015,1.0,Good,Fair


In [ ]:

# Identify changes in construction and reconstruction information.
bridge_history["previous_built_year"] = (
    bridge_history.groupby("bridge_id")["YEAR_BUILT_027"].shift(1)
)

bridge_history["previous_reconstructed_year"] = (
    bridge_history.groupby("bridge_id")["YEAR_RECONSTRUCTED_106"].shift(1)
)

# Detect changes in reported construction year.
built_year_change = (
    bridge_history["previous_built_year"].notna() &
    bridge_history["YEAR_BUILT_027"].notna() &
    (
        bridge_history["YEAR_BUILT_027"] !=
        bridge_history["previous_built_year"]
    )
)

# Detect changes in reported reconstruction year.
reconstruction_year_change = (
    bridge_history["previous_reconstructed_year"].notna() &
    bridge_history["YEAR_RECONSTRUCTED_106"].notna() &
    (
        bridge_history["YEAR_RECONSTRUCTED_106"] !=
        bridge_history["previous_reconstructed_year"]
    )
)

# Identify reported reconstructions occurring within the observation period.
reconstruction_during_history = (
    bridge_history["YEAR_RECONSTRUCTED_106"].notna() &
    (
        bridge_history["YEAR_RECONSTRUCTED_106"] >=
        bridge_history["observation_year"] - 1
    ) &
    (
        bridge_history["YEAR_RECONSTRUCTED_106"] <=
        bridge_history["observation_year"]
    )
)

print(
    "Bridges with changes in reported construction year:",
    f"{bridge_history.loc[built_year_change, 'bridge_id'].nunique():,}"
)

print(
    "Bridges with changes in reported reconstruction year:",
    f"{bridge_history.loc[reconstruction_year_change, 'bridge_id'].nunique():,}"
)

print(
    "Bridges with reconstruction reported near an observation year:",
    f"{bridge_history.loc[reconstruction_during_history, 'bridge_id'].nunique():,}"
)

# Display example construction year changes.
display(
    bridge_history.loc[
        built_year_change,
        [
            "bridge_id",
            "previous_year",
            "observation_year",
            "previous_built_year",
            "YEAR_BUILT_027",
            "previous_condition",
            "condition"
        ]
    ].head(10)
)


Bridges with changes in reported construction year: 81,251
Bridges with changes in reported reconstruction year: 94,787
Bridges with reconstruction reported near an observation year: 36,849


,bridge_id,previous_year,observation_year,previous_built_year,YEAR_BUILT_027,previous_condition,condition
12825,01_000002,2000.0,2001,1900.0,1898.0,Poor,Poor
14301,01_000006,2000.0,2001,1900.0,1940.0,Poor,Poor
14342,01_000007,1993.0,1994,1900.0,1993.0,Poor,Good
14385,01_000008,2000.0,2002,1900.0,1898.0,Fair,Fair
14387,01_000008,2003.0,2004,1898.0,1927.0,Fair,Poor
15872,01_000014,1995.0,1996,1900.0,1910.0,Poor,Poor
16189,01_000036,1996.0,1997,1900.0,1996.0,Poor,Poor
16516,01_000050,2022.0,2023,1910.0,1995.0,Fair,Fair
16634,01_000060,1995.0,1996,1912.0,1994.0,Poor,Good
16809,01_000071,2024.0,2025,1913.0,1992.0,Fair,Fair


In [ ]:

# Examine the distribution of reported reconstruction years.
reconstruction_values = bridge_history["YEAR_RECONSTRUCTED_106"]

print("Most common reconstruction year values:")
display(
    reconstruction_values.value_counts(
        dropna=False
    ).head(15)
)

# Identify potentially valid reconstruction years.
valid_reconstruction = (
    reconstruction_values.between(1800, 2025) &
    (
        reconstruction_values <=
        bridge_history["observation_year"]
    )
)

print(
    "\nObservations with potentially valid reconstruction years:",
    f"{valid_reconstruction.sum():,}"
)

print(
    "Unique bridges with potentially valid reconstruction years:",
    f"{bridge_history.loc[valid_reconstruction, 'bridge_id'].nunique():,}"
)

# Check how reconstruction years compare with construction years.
reconstruction_sample = bridge_history.loc[
    valid_reconstruction,
    [
        "bridge_id",
        "observation_year",
        "YEAR_BUILT_027",
        "YEAR_RECONSTRUCTED_106",
        "condition"
    ]
]

display(reconstruction_sample.head(10))


Most common reconstruction year values:


YEAR_RECONSTRUCTED_106
0.0       16808254
NaN         913650
1988.0       87812
1990.0       85409
1987.0       83958
1989.0       83840
1992.0       82639
1991.0       81315
1986.0       80724
1985.0       79439
1984.0       72247
1993.0       71599
1994.0       67975
1995.0       63757
1980.0       63231
Name: count, dtype: int64


Observations with potentially valid reconstruction years: 2,918,805
Unique bridges with potentially valid reconstruction years: 180,223


,bridge_id,observation_year,YEAR_BUILT_027,YEAR_RECONSTRUCTED_106,condition
86,01_0000000001034Z0,1992,1940.0,1978.0,Poor
87,01_0000000001035Z0,1992,1940.0,1978.0,Fair
90,01_0000000001074Z0,1992,1935.0,1980.0,Good
106,01_0000000002131Z0,1992,1932.0,1980.0,Fair
113,01_0000000002168Z0,1992,1952.0,1985.0,Fair
177,01_0000000003166Z0,1992,1940.0,1990.0,Fair
180,01_0000000007058Z0,1992,1956.0,1972.0,Fair
188,01_0000000007100Z0,1992,1950.0,1971.0,Fair
189,01_0000000007101Z0,1992,1952.0,1975.0,Good
192,01_0000000007108Z0,1992,1978.0,1990.0,Good


In [ ]:

# Treat zero and missing reconstruction years as unavailable.
reconstruction_year = bridge_history["YEAR_RECONSTRUCTED_106"].where(
    bridge_history["YEAR_RECONSTRUCTED_106"].between(1800, 2025)
)

# Identify the first recorded reconstruction year for each bridge.
bridge_history["valid_reconstruction_year"] = reconstruction_year

# Detect newly reported reconstructions during consecutive observations.
previous_reconstruction = (
    bridge_history.groupby("bridge_id")["valid_reconstruction_year"].shift(1)
)

new_reconstruction = (
    bridge_history["previous_year"].notna() &
    bridge_history["valid_reconstruction_year"].notna() &
    (
        bridge_history["valid_reconstruction_year"] >
        bridge_history["previous_year"]
    ) &
    (
        bridge_history["valid_reconstruction_year"] <=
        bridge_history["observation_year"]
    ) &
    (
        previous_reconstruction.isna() |
        (
            bridge_history["valid_reconstruction_year"] !=
            previous_reconstruction
        )
    )
)

# Summarise reconstruction activity during recorded follow-up.
print(
    "Newly reported reconstruction observations:",
    f"{new_reconstruction.sum():,}"
)

print(
    "Unique bridges with newly reported reconstructions:",
    f"{bridge_history.loc[new_reconstruction, 'bridge_id'].nunique():,}"
)

print("\nExamples:")
display(
    bridge_history.loc[
        new_reconstruction,
        [
            "bridge_id",
            "previous_year",
            "observation_year",
            "previous_reconstructed_year",
            "valid_reconstruction_year",
            "previous_condition",
            "condition"
        ]
    ].head(10)
)


Newly reported reconstruction observations: 2,402
Unique bridges with newly reported reconstructions: 2,388

Examples:


,bridge_id,previous_year,observation_year,previous_reconstructed_year,valid_reconstruction_year,previous_condition,condition
19293,01_000186,2021.0,2022,0.0,2022.0,Poor,Poor
44200,01_001339,1993.0,1994,0.0,1994.0,Fair,Good
126013,01_004743,2011.0,2012,0.0,2012.0,Poor,Good
133947,01_005052,1994.0,1995,0.0,1995.0,Poor,Good
152768,01_005788,1994.0,1995,0.0,1995.0,Fair,Good
154123,01_005841,2003.0,2004,0.0,2004.0,Fair,Good
168698,01_006366,1994.0,1995,0.0,1995.0,Good,Good
186516,01_007017,1994.0,1995,0.0,1995.0,Fair,Good
194901,01_007320,1996.0,1998,0.0,1997.0,Poor,Fair
233543,01_008666,1995.0,1996,0.0,1996.0,Poor,Good


In [ ]:

# Select observations where the bridge is classified as Good.
good_observations = bridge_history.loc[
    bridge_history["condition"] == "Good"
].copy()

# Identify the earliest observed Good condition for each bridge.
first_good = (
    good_observations
    .groupby("bridge_id", as_index=False)
    .first()
)

# Retain the baseline variables required for survival analysis.
baseline_columns = [
    "bridge_id",
    "observation_year",
    "bridge_age",
    "YEAR_BUILT_027",
    "YEAR_RECONSTRUCTED_106",
    "ADT_029",
    "STRUCTURE_KIND_043A",
    "STRUCTURE_TYPE_043B",
    "STRUCTURE_LEN_MT_049"
]

first_good = first_good[baseline_columns].rename(
    columns={"observation_year": "start_year"}
)

# Summarise eligible baseline observations.
print(
    "Bridges with at least one Good observation:",
    f"{len(first_good):,}"
)

print("\nStarting year distribution:")
display(first_good["start_year"].describe())

print("\nBaseline bridge age distribution:")
display(first_good["bridge_age"].describe())

print("\nMissing baseline predictor values:")
display(
    first_good[
        [
            "bridge_age",
            "ADT_029",
            "STRUCTURE_LEN_MT_049",
            "STRUCTURE_KIND_043A",
            "STRUCTURE_TYPE_043B"
        ]
    ].isna().sum()
)


Bridges with at least one Good observation: 700,637

Starting year distribution:


count    700637.000000
mean       2000.015586
std           9.657327
min        1992.000000
25%        1992.000000
50%        1995.000000
75%        2006.000000
max        2025.000000
Name: start_year, dtype: float64


Baseline bridge age distribution:


count    700280.000000
mean         21.744929
std          22.207562
min           0.000000
25%           2.000000
50%          15.000000
75%          34.000000
max         200.000000
Name: bridge_age, dtype: float64


Missing baseline predictor values:


bridge_age              357
ADT_029                 710
STRUCTURE_LEN_MT_049    393
STRUCTURE_KIND_043A     111
STRUCTURE_TYPE_043B     432
dtype: int64

In [ ]:

# Identify the first recorded observation for each bridge.
first_observation = (
    bridge_history
    .groupby("bridge_id", as_index=False)
    .first()
)

# Count bridges by their first recorded condition.
print("First recorded condition distribution:")
display(
    first_observation["condition"].value_counts()
)

# Select bridges initially observed in Good condition.
eligible_bridges = first_observation.loc[
    first_observation["condition"] == "Good",
    "bridge_id"
]

# Extract the corresponding baseline records.
eligible_baselines = first_good.loc[
    first_good["bridge_id"].isin(eligible_bridges)
].copy()

print(
    "\nBridges initially observed as Good:",
    f"{len(eligible_baselines):,}"
)

print(
    "Bridges first observed as Fair/Poor/Unknown, then Good:",
    f"{len(first_good) - len(eligible_baselines):,}"
)

print("\nEligible baseline year distribution:")
display(
    eligible_baselines["start_year"].describe()
)


First recorded condition distribution:


condition
Good       557716
Fair       342624
Poor       129006
Unknown      9761
Name: count, dtype: int64


Bridges initially observed as Good: 557,716
Bridges first observed as Fair/Poor/Unknown, then Good: 142,921

Eligible baseline year distribution:


count    557716.000000
mean       1999.191366
std           9.695822
min        1992.000000
25%        1992.000000
50%        1993.000000
75%        2006.000000
max        2025.000000
Name: start_year, dtype: float64

In [ ]:

from zipfile import ZipFile, ZIP_DEFLATED

# Package the cleaned annual datasets for group sharing.
output_zip = PROJECT / "nbi_cleaned_data_v1.zip"

with ZipFile(output_zip, "w", compression=ZIP_DEFLATED) as archive:
    for file_path in sorted(PROCESSED.glob("nbi_*_clean.parquet")):
        archive.write(
            file_path,
            arcname=f"processed/{file_path.name}"
        )

print("Archive created:", output_zip)
print("Number of datasets:", len(list(PROCESSED.glob("nbi_*_clean.parquet"))))
print("Archive size (GB):", round(output_zip.stat().st_size / 1e9, 2))



Archive created: /Users/annie/Downloads/bridge-deterioration/nbi_cleaned_data_v1.zip
Number of datasets: 34
Archive size (GB): 0.31


In [ ]:

# Retain historical observations for eligible bridges.
eligible_history = bridge_history.loc[
    bridge_history["bridge_id"].isin(eligible_baselines["bridge_id"])
].copy()

# Identify the next recorded observation within each bridge history.
eligible_history["next_year"] = (
    eligible_history.groupby("bridge_id")["observation_year"].shift(-1)
)

# Identify missing annual observations.
eligible_history["next_year_gap"] = (
    eligible_history["next_year"] -
    eligible_history["observation_year"]
)

# Identify potential deterioration events.
eligible_history["is_deterioration"] = (
    eligible_history["condition"].isin(["Fair", "Poor"])
)

# Identify unknown condition observations.
eligible_history["is_unknown"] = (
    eligible_history["condition"] == "Unknown"
)

# Identify possible changes to the physical bridge.
eligible_history["is_structure_change"] = (
    built_year_change.reindex(eligible_history.index, fill_value=False) |
    new_reconstruction.reindex(eligible_history.index, fill_value=False)
)

# Identify the first observation requiring follow-up to stop.
eligible_history["is_stop"] = (
    eligible_history["is_deterioration"] |
    eligible_history["is_unknown"] |
    eligible_history["is_structure_change"] |
    (eligible_history["next_year_gap"] > 1)
)

# Summarise potential stopping conditions.
print("Eligible bridge-year observations:", f"{len(eligible_history):,}")
print("Potential deterioration observations:", f"{eligible_history['is_deterioration'].sum():,}")
print("Unknown condition observations:", f"{eligible_history['is_unknown'].sum():,}")
print("Potential structure changes:", f"{eligible_history['is_structure_change'].sum():,}")
print("Observation gaps:", f"{(eligible_history['next_year_gap'] > 1).sum():,}")


Eligible bridge-year observations: 11,530,704
Potential deterioration observations: 3,592,515
Unknown condition observations: 6,714
Potential structure changes: 25,104
Observation gaps: 8,522


In [ ]:

# Identify the first stopping observation for each eligible bridge.
stop_records = eligible_history.loc[
    eligible_history["is_stop"]
].copy()

first_stop = (
    stop_records
    .sort_values(["bridge_id", "observation_year"])
    .drop_duplicates(subset="bridge_id", keep="first")
)

# Classify the reason for ending follow-up.
first_stop["stop_reason"] = np.select(
    [
        first_stop["is_structure_change"],
        first_stop["is_unknown"],
        first_stop["next_year_gap"] > 1,
        first_stop["is_deterioration"]
    ],
    [
        "Structure change",
        "Unknown condition",
        "Observation gap",
        "Deterioration"
    ],
    default="Other"
)

# Use the preceding observation for structure changes and unknown conditions.
first_stop["end_year"] = np.where(
    first_stop["stop_reason"].isin(
        ["Structure change", "Unknown condition"]
    ),
    first_stop["previous_year"],
    first_stop["observation_year"]
)

# Classify observed deterioration events.
first_stop["event"] = (
    first_stop["stop_reason"] == "Deterioration"
).astype(int)

print("Bridges with a stopping condition:", f"{len(first_stop):,}")

print("\nFirst stopping reason:")
display(first_stop["stop_reason"].value_counts())

print("\nObserved deterioration events:")
print(f"{first_stop['event'].sum():,}")

print("\nExamples:")
display(
    first_stop[
        ["bridge_id", "observation_year", "end_year",
         "stop_reason", "event"]
    ].head(10)
)


Bridges with a stopping condition: 288,704

First stopping reason:


stop_reason
Deterioration        266133
Structure change      13843
Observation gap        6059
Unknown condition      2669
Name: count, dtype: int64


Observed deterioration events:
266,133

Examples:


,bridge_id,observation_year,end_year,stop_reason,event
3,01_00000000000S700,1999,1999.0,Deterioration,1
33,01_00000000000S702,2022,2022.0,Deterioration,1
53,01_00000000000S703,2022,2022.0,Deterioration,1
65,01_0000000000M0022,2006,2006.0,Deterioration,1
10755,01_000000883039900,1992,1992.0,Observation gap,0
11174,01_000001014002450,1992,1992.0,Observation gap,0
14016,01_000004504800350,1992,1992.0,Observation gap,0
14452,01_00000B253040000,1992,1992.0,Observation gap,0
14501,01_00000BLN0BR0010,2015,2015.0,Deterioration,1
15785,01_000011,1995,1995.0,Deterioration,1


In [ ]:

# Prioritise observed deterioration over gaps following the event.
first_stop["stop_reason"] = np.select(
    [
        first_stop["is_structure_change"],
        first_stop["is_unknown"],
        first_stop["is_deterioration"],
        first_stop["next_year_gap"] > 1
    ],
    [
        "Structure change",
        "Unknown condition",
        "Deterioration",
        "Observation gap"
    ],
    default="Other"
)

# End follow-up before an observed structure change or unknown condition.
first_stop["end_year"] = np.where(
    first_stop["stop_reason"].isin(
        ["Structure change", "Unknown condition"]
    ),
    first_stop["previous_year"],
    first_stop["observation_year"]
)

# Record deterioration as the event of interest.
first_stop["event"] = (
    first_stop["stop_reason"] == "Deterioration"
).astype(int)

# Attach the baseline year for validation.
stop_validation = first_stop[
    ["bridge_id", "end_year", "event", "stop_reason"]
].merge(
    eligible_baselines[["bridge_id", "start_year"]],
    on="bridge_id",
    how="left",
    validate="one_to_one"
)

# Calculate the observed follow-up duration.
stop_validation["duration"] = (
    stop_validation["end_year"] -
    stop_validation["start_year"]
)

print("Updated stopping reasons:")
display(stop_validation["stop_reason"].value_counts())

print("\nInvalid negative durations:",
      (stop_validation["duration"] < 0).sum())

print("Zero-duration records:",
      (stop_validation["duration"] == 0).sum())

print("Missing end years:",
      stop_validation["end_year"].isna().sum())

print("\nDuration summary:")
display(stop_validation["duration"].describe())


Updated stopping reasons:


stop_reason
Deterioration        266298
Structure change      13843
Observation gap        5894
Unknown condition      2669
Name: count, dtype: int64


Invalid negative durations: 0
Zero-duration records: 5222
Missing end years: 0

Duration summary:


count    288704.000000
mean         10.277606
std           8.296117
min           0.000000
25%           3.000000
50%           8.000000
75%          16.000000
max          33.000000
Name: duration, dtype: float64

In [ ]:

# Examine records with zero observed follow-up duration.
zero_duration = stop_validation.loc[
    stop_validation["duration"] == 0
].copy()

print("Total zero-duration records:", f"{len(zero_duration):,}")

print("\nZero-duration records by stopping reason:")
display(zero_duration["stop_reason"].value_counts())

print("\nZero-duration deterioration events:")
print(f"{zero_duration['event'].sum():,}")

print("\nExamples:")
display(
    zero_duration[
        [
            "bridge_id",
            "start_year",
            "end_year",
            "duration",
            "event",
            "stop_reason"
        ]
    ].head(15)
)


Total zero-duration records: 5,222

Zero-duration records by stopping reason:


stop_reason
Structure change     3047
Observation gap      1853
Unknown condition     322
Name: count, dtype: int64


Zero-duration deterioration events:
0

Examples:


,bridge_id,start_year,end_year,duration,event,stop_reason
4,01_000000883039900,1992,1992.0,0.0,0,Observation gap
5,01_000001014002450,1992,1992.0,0.0,0,Observation gap
6,01_000004504800350,1992,1992.0,0.0,0,Observation gap
7,01_00000B253040000,1992,1992.0,0.0,0,Observation gap
10,01_000013771700120,1992,1992.0,0.0,0,Observation gap
839,01_003810,1993,1993.0,0.0,0,Observation gap
1217,01_005055,1993,1993.0,0.0,0,Structure change
2477,01_008459,1993,1993.0,0.0,0,Unknown condition
3953,01_011526,1995,1995.0,0.0,0,Structure change
5566,01_014314,1993,1993.0,0.0,0,Structure change


In [ ]:

# Identify the final recorded observation year for each eligible bridge.
last_observation = (
    eligible_history.groupby("bridge_id")["observation_year"]
    .max()
    .rename("last_year")
    .reset_index()
)

# Combine baseline characteristics with stopping information.
survival_data = eligible_baselines.merge(
    first_stop[
        ["bridge_id", "end_year", "event", "stop_reason"]
    ],
    on="bridge_id",
    how="left",
    validate="one_to_one"
)

# Attach the final recorded year for bridges without a stopping point.
survival_data = survival_data.merge(
    last_observation,
    on="bridge_id",
    how="left",
    validate="one_to_one"
)

# Right-censor bridges without an earlier stopping condition.
survival_data["end_year"] = survival_data["end_year"].fillna(
    survival_data["last_year"]
)

survival_data["event"] = (
    survival_data["event"].fillna(0).astype(int)
)

survival_data["stop_reason"] = (
    survival_data["stop_reason"].fillna("End of observation")
)

# Calculate time from the baseline Good observation to the endpoint.
survival_data["duration"] = (
    survival_data["end_year"] - survival_data["start_year"]
)

# Exclude observations without positive follow-up time.
zero_duration_count = (survival_data["duration"] == 0).sum()

survival_data = survival_data.loc[
    survival_data["duration"] > 0
].copy()

# Summarise the constructed survival dataset.
print("Survival records:", f"{len(survival_data):,}")
print("Zero-duration records excluded:", f"{zero_duration_count:,}")

print("\nEvent distribution:")
display(survival_data["event"].value_counts())

print("\nStopping reasons:")
display(survival_data["stop_reason"].value_counts())

print("\nDuration summary:")
display(survival_data["duration"].describe())

print("\nSample records:")
display(survival_data.head(10))


Survival records: 521,563
Zero-duration records excluded: 36,153

Event distribution:


event
1    266298
0    255265
Name: count, dtype: int64


Stopping reasons:


stop_reason
Deterioration         266298
End of observation    238081
Structure change       10796
Observation gap         4041
Unknown condition       2347
Name: count, dtype: int64


Duration summary:


count    521563.000000
mean         12.742094
std           9.631264
min           1.000000
25%           5.000000
50%          10.000000
75%          19.000000
max          33.000000
Name: duration, dtype: float64


Sample records:


,bridge_id,start_year,bridge_age,YEAR_BUILT_027,YEAR_RECONSTRUCTED_106,ADT_029,STRUCTURE_KIND_043A,STRUCTURE_TYPE_043B,STRUCTURE_LEN_MT_049,end_year,event,stop_reason,last_year,duration
0,01_00000000000S700,1996,7.0,1989.0,0.0,80.0,5,01,11.0,1999.0,1,Deterioration,2005,3.0
1,01_00000000000S702,2003,4.0,1999.0,0.0,50.0,5,05,14.7,2022.0,1,Deterioration,2025,19.0
2,01_00000000000S703,2006,4.0,2002.0,0.0,150.0,5,01,13.7,2022.0,1,Deterioration,2025,16.0
3,01_0000000000M0022,1998,56.0,1942.0,0.0,800.0,1,02,43.3,2006.0,1,Deterioration,2025,8.0
7189,01_00000BLN0BR0010,2012,38.0,1974.0,0.0,250.0,1,19,6.4,2015.0,1,Deterioration,2025,3.0
8105,01_000011,1993,93.0,1900.0,0.0,75.0,7,02,6.1,1995.0,1,Deterioration,1996,2.0
8107,01_000042,1993,83.0,1910.0,0.0,18470.0,1,19,28.0,1995.0,1,Deterioration,2025,2.0
8108,01_000071,1993,80.0,1913.0,1992.0,45.0,3,02,24.4,2004.0,1,Deterioration,2025,11.0
8109,01_000093,1993,78.0,1915.0,0.0,250.0,1,19,6.7,2003.0,1,Deterioration,2025,10.0
8263,01_000108,1993,0.0,1993.0,0.0,33.0,1,19,6.4,2003.0,1,Deterioration,2025,10.0


In [ ]:

# Check the structural integrity of the survival dataset.
print("Total survival records:", f"{len(survival_data):,}")

print("Duplicate bridge identifiers:",
      survival_data["bridge_id"].duplicated().sum())

print("Missing durations:",
      survival_data["duration"].isna().sum())

print("Non-positive durations:",
      (survival_data["duration"] <= 0).sum())

print("Invalid event values:",
      (~survival_data["event"].isin([0, 1])).sum())

print("Endpoints before baseline:",
      (survival_data["end_year"] < survival_data["start_year"]).sum())

print("Endpoints after final observation:",
      (survival_data["end_year"] > survival_data["last_year"]).sum())

# Check that baseline bridge ages agree with reported construction years.
age_mismatch = (
    survival_data["bridge_age"].notna() &
    survival_data["YEAR_BUILT_027"].notna() &
    (
        survival_data["bridge_age"] !=
        survival_data["start_year"] -
        survival_data["YEAR_BUILT_027"]
    )
)

print("Baseline age mismatches:", age_mismatch.sum())

# Check the distribution of survival endpoints.
print("\nEndpoint years:")
display(survival_data["end_year"].describe())

print("\nMissing predictor values:")
display(
    survival_data[
        [
            "bridge_age",
            "ADT_029",
            "STRUCTURE_LEN_MT_049",
            "STRUCTURE_KIND_043A",
            "STRUCTURE_TYPE_043B"
        ]
    ].isna().sum()
)


Total survival records: 521,563
Duplicate bridge identifiers: 0
Missing durations: 0
Non-positive durations: 0
Invalid event values: 0
Endpoints before baseline: 0
Endpoints after final observation: 0
Baseline age mismatches: 1233

Endpoint years:


count    521563.000000
mean       2012.004389
std          11.639427
min        1993.000000
25%        2001.000000
50%        2013.000000
75%        2025.000000
max        2025.000000
Name: end_year, dtype: float64


Missing predictor values:


bridge_age              227
ADT_029                 270
STRUCTURE_LEN_MT_049     55
STRUCTURE_KIND_043A      33
STRUCTURE_TYPE_043B      49
dtype: int64

In [ ]:

# Examine records with inconsistent baseline age calculations.
age_mismatch_records = survival_data.loc[
    age_mismatch,
    [
        "bridge_id",
        "start_year",
        "bridge_age",
        "YEAR_BUILT_027",
        "duration",
        "event"
    ]
].copy()

# Calculate the expected age from the baseline year and construction year.
age_mismatch_records["expected_age"] = (
    age_mismatch_records["start_year"] -
    age_mismatch_records["YEAR_BUILT_027"]
)

age_mismatch_records["age_difference"] = (
    age_mismatch_records["bridge_age"] -
    age_mismatch_records["expected_age"]
)

print("Total age mismatches:", f"{len(age_mismatch_records):,}")

print("\nAge difference distribution:")
display(
    age_mismatch_records["age_difference"].value_counts().head(15)
)

print("\nExamples:")
display(age_mismatch_records.head(15))

# Check whether baseline records match the original historical observations.
baseline_check = survival_data[
    ["bridge_id", "start_year", "bridge_age", "YEAR_BUILT_027"]
].merge(
    bridge_history[
        ["bridge_id", "observation_year", "bridge_age", "YEAR_BUILT_027"]
    ],
    left_on=["bridge_id", "start_year"],
    right_on=["bridge_id", "observation_year"],
    how="left",
    suffixes=("_baseline", "_history"),
    validate="one_to_one"
)

print("\nBaseline records without matching historical observations:")
print(baseline_check["observation_year"].isna().sum())


Total age mismatches: 1,233

Age difference distribution:


age_difference
1.0     428
2.0     222
3.0      86
4.0      50
5.0      47
6.0      37
7.0      33
8.0      19
13.0     16
19.0     13
9.0      12
43.0     12
16.0      9
53.0      9
58.0      8
Name: count, dtype: int64


Examples:


,bridge_id,start_year,bridge_age,YEAR_BUILT_027,duration,event,expected_age,age_difference
21498,02_100210000001295,1995,37.0,1975.0,9.0,0,20.0,17.0
21550,02_100310000001052,1995,19.0,1995.0,7.0,0,0.0,19.0
21561,02_100320000001064,1995,25.0,1989.0,10.0,1,6.0,19.0
21574,02_100320000001340,1995,25.0,1994.0,10.0,1,1.0,24.0
21581,02_100320000001390,1995,16.0,1994.0,10.0,1,1.0,15.0
21582,02_100320000001391,1995,15.0,1994.0,30.0,0,1.0,14.0
21583,02_100320000001392,1995,15.0,1994.0,23.0,1,1.0,14.0
21584,02_100320000001393,1995,15.0,1994.0,18.0,1,1.0,14.0
21585,02_100320000001394,1995,15.0,1994.0,30.0,0,1.0,14.0
21586,02_100320000001395,1995,15.0,1994.0,30.0,0,1.0,14.0



Baseline records without matching historical observations:
0


In [ ]:

# Select the actual first Good observation for each eligible bridge.
corrected_baselines = (
    bridge_history.loc[
        (bridge_history["bridge_id"].isin(eligible_bridges)) &
        (bridge_history["condition"] == "Good")
    ]
    .sort_values(["bridge_id", "observation_year"])
    .drop_duplicates(subset="bridge_id", keep="first")
    [baseline_columns]
    .rename(columns={"observation_year": "start_year"})
    .copy()
)

# Compare the corrected baseline values with the existing survival dataset.
baseline_comparison = survival_data[
    ["bridge_id", "start_year", "bridge_age", "YEAR_BUILT_027"]
].merge(
    corrected_baselines[
        ["bridge_id", "start_year", "bridge_age", "YEAR_BUILT_027"]
    ],
    on="bridge_id",
    how="left",
    suffixes=("_old", "_corrected"),
    validate="one_to_one"
)

# Check age consistency in the corrected baseline records.
corrected_age_mismatch = (
    corrected_baselines["bridge_age"].notna() &
    corrected_baselines["YEAR_BUILT_027"].notna() &
    (
        corrected_baselines["bridge_age"] !=
        corrected_baselines["start_year"] -
        corrected_baselines["YEAR_BUILT_027"]
    )
)

print("Corrected baseline records:", f"{len(corrected_baselines):,}")
print("Corrected baseline age mismatches:", corrected_age_mismatch.sum())

print("\nBaseline years changed:")
print(
    (
        baseline_comparison["start_year_old"] !=
        baseline_comparison["start_year_corrected"]
    ).sum()
)

print("\nExamples of corrected baseline ages:")
display(
    baseline_comparison.loc[
        baseline_comparison["bridge_age_old"] !=
        baseline_comparison["bridge_age_corrected"],
        [
            "bridge_id",
            "start_year_old",
            "bridge_age_old",
            "bridge_age_corrected",
            "YEAR_BUILT_027_old",
            "YEAR_BUILT_027_corrected"
        ]
    ].head(10)
)


Corrected baseline records: 557,716
Corrected baseline age mismatches: 0

Baseline years changed:
0

Examples of corrected baseline ages:


,bridge_id,start_year_old,bridge_age_old,bridge_age_corrected,YEAR_BUILT_027_old,YEAR_BUILT_027_corrected
12589,01_021780,2024,NaN,NaN,1800.0,1800.0
12590,01_021785,2024,NaN,NaN,1800.0,1800.0
12591,01_021787,2024,NaN,NaN,1800.0,1800.0
12592,01_021795,2024,NaN,NaN,1800.0,1800.0
13116,02_100210000001295,1995,37.0,NaN,1975.0,NaN
13167,02_100310000001052,1995,19.0,NaN,1995.0,NaN
13177,02_100320000000902,1998,NaN,NaN,NaN,NaN
13178,02_100320000001064,1995,25.0,NaN,1989.0,NaN
13191,02_100320000001340,1995,25.0,NaN,1994.0,NaN
13198,02_100320000001390,1995,16.0,NaN,1994.0,NaN


In [ ]:

# Retain the previously constructed survival outcomes.
outcome_columns = [
    "bridge_id",
    "end_year",
    "event",
    "stop_reason",
    "last_year",
    "duration"
]

survival_outcomes = survival_data[outcome_columns].copy()

# Rebuild the survival dataset using corrected baseline observations.
survival_data = corrected_baselines.merge(
    survival_outcomes,
    on="bridge_id",
    how="inner",
    validate="one_to_one"
)

# Verify the corrected survival dataset.
age_mismatch = (
    survival_data["bridge_age"].notna() &
    survival_data["YEAR_BUILT_027"].notna() &
    (
        survival_data["bridge_age"] !=
        survival_data["start_year"] -
        survival_data["YEAR_BUILT_027"]
    )
)

print("Survival records:", f"{len(survival_data):,}")
print("Age mismatches:", age_mismatch.sum())
print("Duplicate bridge identifiers:",
      survival_data["bridge_id"].duplicated().sum())

print("\nEvent distribution:")
display(survival_data["event"].value_counts())

print("\nMissing baseline predictors:")
display(
    survival_data[
        [
            "bridge_age",
            "ADT_029",
            "STRUCTURE_LEN_MT_049",
            "STRUCTURE_KIND_043A",
            "STRUCTURE_TYPE_043B"
        ]
    ].isna().sum()
)


Survival records: 521,563
Age mismatches: 0
Duplicate bridge identifiers: 0

Event distribution:


event
1    266298
0    255265
Name: count, dtype: int64


Missing baseline predictors:


bridge_age              1460
ADT_029                 2012
STRUCTURE_LEN_MT_049     230
STRUCTURE_KIND_043A      348
STRUCTURE_TYPE_043B      389
dtype: int64

In [ ]:

# Inspect condition-rating availability in the cleaned annual datasets.
condition_check = []

for year in range(1992, 2026):
    df_year = pd.read_parquet(
        PROCESSED / f"nbi_{year}_clean.parquet",
        columns=[
            "DECK_COND_058",
            "SUPERSTRUCTURE_COND_059",
            "SUBSTRUCTURE_COND_060",
            "CULVERT_COND_062"
        ]
    )

    component_available = df_year[
        [
            "DECK_COND_058",
            "SUPERSTRUCTURE_COND_059",
            "SUBSTRUCTURE_COND_060"
        ]
    ].notna().any(axis=1)

    culvert_available = df_year["CULVERT_COND_062"].notna()

    condition_check.append({
        "year": year,
        "both_available": (
            component_available & culvert_available
        ).sum(),
        "culvert_only": (
            ~component_available & culvert_available
        ).sum(),
        "components_only": (
            component_available & ~culvert_available
        ).sum(),
        "neither_available": (
            ~component_available & ~culvert_available
        ).sum()
    })

condition_check = pd.DataFrame(condition_check)

print("Condition-rating availability across all years:")
display(condition_check.sum(numeric_only=True))

print("\nSelected years:")
display(
    condition_check.loc[
        condition_check["year"].isin([1992, 2000, 2010, 2020, 2025])
    ]
)


Condition-rating availability across all years:


year                    68289
both_available          30523
culvert_only          4419889
components_only      16165661
neither_available       30271
dtype: int64


Selected years:


,year,both_available,culvert_only,components_only,neither_available
0,1992,1805,106294,475807,920
8,2000,1475,120371,475341,2405
18,2010,104,130770,473167,377
28,2020,21,143017,475416,0
33,2025,82,148472,475635,0


In [ ]:

# Compare component-based and culvert-based condition classifications.
condition_conflicts = []

for year in range(1992, 2026):
    df_year = pd.read_parquet(
        PROCESSED / f"nbi_{year}_clean.parquet",
        columns=[
            "bridge_id",
            "DECK_COND_058",
            "SUPERSTRUCTURE_COND_059",
            "SUBSTRUCTURE_COND_060",
            "CULVERT_COND_062"
        ]
    )

    component_rating = df_year[
        [
            "DECK_COND_058",
            "SUPERSTRUCTURE_COND_059",
            "SUBSTRUCTURE_COND_060"
        ]
    ].min(axis=1)

    culvert_rating = df_year["CULVERT_COND_062"]

    # Restrict the comparison to observations with both rating types.
    overlap = component_rating.notna() & culvert_rating.notna()

    # Convert numerical ratings into condition categories.
    def classify_condition(rating):
        return np.select(
            [
                rating.between(7, 9),
                rating.between(5, 6),
                rating.between(0, 4)
            ],
            ["Good", "Fair", "Poor"],
            default="Unknown"
        )

    component_condition = classify_condition(component_rating)
    culvert_condition = classify_condition(culvert_rating)

    conflicts = overlap & (component_condition != culvert_condition)

    condition_conflicts.append({
        "year": year,
        "overlapping_records": int(overlap.sum()),
        "conflicting_classifications": int(conflicts.sum())
    })

condition_conflicts = pd.DataFrame(condition_conflicts)

print("Total overlapping records:",
      f"{condition_conflicts['overlapping_records'].sum():,}")

print("Different condition classifications:",
      f"{condition_conflicts['conflicting_classifications'].sum():,}")

print("\nYears with the most classification conflicts:")
display(
    condition_conflicts.sort_values(
        "conflicting_classifications",
        ascending=False
    ).head(10)
)


Total overlapping records: 30,523
Different condition classifications: 6,441

Years with the most classification conflicts:


,year,overlapping_records,conflicting_classifications
5,1997,2633,722
6,1998,2716,710
1,1993,2036,492
0,1992,1805,491
7,1999,1995,459
3,1995,1968,450
2,1994,1924,438
4,1996,1934,389
11,2003,1491,261
15,2007,1337,246


In [ ]:

# Examine structure types where component and culvert classifications disagree.
conflict_details = []

for year in range(1992, 2026):
    df_year = pd.read_parquet(
        PROCESSED / f"nbi_{year}_clean.parquet",
        columns=[
            "bridge_id",
            "STRUCTURE_KIND_043A",
            "STRUCTURE_TYPE_043B",
            "DECK_COND_058",
            "SUPERSTRUCTURE_COND_059",
            "SUBSTRUCTURE_COND_060",
            "CULVERT_COND_062"
        ]
    )

    component_rating = df_year[
        [
            "DECK_COND_058",
            "SUPERSTRUCTURE_COND_059",
            "SUBSTRUCTURE_COND_060"
        ]
    ].min(axis=1)

    culvert_rating = df_year["CULVERT_COND_062"]

    overlap = component_rating.notna() & culvert_rating.notna()

    conflicts = overlap & (
        classify_condition(component_rating) !=
        classify_condition(culvert_rating)
    )

    if conflicts.any():
        details = df_year.loc[
            conflicts,
            [
                "bridge_id",
                "STRUCTURE_KIND_043A",
                "STRUCTURE_TYPE_043B"
            ]
        ].copy()

        details["year"] = year
        conflict_details.append(details)

conflict_details = pd.concat(conflict_details, ignore_index=True)

print("Conflicting observations:", f"{len(conflict_details):,}")
print("Unique affected bridge IDs:",
      f"{conflict_details['bridge_id'].nunique():,}")

print("\nStructure kind and type combinations:")
display(
    conflict_details.groupby(
        ["STRUCTURE_KIND_043A", "STRUCTURE_TYPE_043B"],
        dropna=False
    ).size().sort_values(ascending=False).head(20)
)

print("\nExamples:")
display(conflict_details.head(10))


Conflicting observations: 6,441
Unique affected bridge IDs: 2,334

Structure kind and type combinations:


STRUCTURE_KIND_043A  STRUCTURE_TYPE_043B
2                    19                     1881
1                    19                      992
3                    02                      734
                     19                      500
5                    05                      261
1                    01                      248
                     04                      180
5                    02                      135
1                    11                      125
4                    02                      106
1                    02                      105
5                    01                      101
1                    20                       96
2                    01                       89
7                    02                       85
1                    07                       54
7                    01                       48
                     19                       46
3                    11                       45
                     10     


Examples:


,bridge_id,STRUCTURE_KIND_043A,STRUCTURE_TYPE_043B,year
0,04_0204,1,01,1992
1,04_0249,2,02,1992
2,04_0980,2,06,1992
3,04_3156,5,02,1992
4,04_3170,1,11,1992
5,04_4212,2,19,1992
6,04_4214,2,19,1992
7,04_4225,2,19,1992
8,04_4981,2,19,1992
9,04_6330,2,19,1992


In [ ]:

# Save the shared survival dataset for all four models.
FINAL.mkdir(parents=True, exist_ok=True)

survival_path = FINAL / "bridge_survival_dataset.parquet"

survival_data.to_parquet(survival_path, index=False)

print("Saved:", survival_path)
print("Records:", f"{len(survival_data):,}")
print("Columns:", len(survival_data.columns))



Saved: /Users/annie/Downloads/bridge-deterioration/data/final/bridge_survival_dataset.parquet
Records: 521,563
Columns: 14
